In [2]:
import os
import numpy as np
import pandas as pd
import time
import MultiSuSiE

# Define ranges for num_causal, LD_BLOCK, and h2_num
num_causal_range = range(1, 4)  # 1 to 3 inclusive
LD_BLOCK_range = range(1, 101)  # 1 to 100 inclusive
h2_num_range = range(1, 3)      # 1 to 2 inclusive

# Loop through num_causal, LD_BLOCK, and h2_num
for num_causal in num_causal_range:
    for LD_BLOCK in LD_BLOCK_range:
        for h2_num in h2_num_range:
            
            # Set directories and file paths
            wrk_dir = f"/scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_1mb/shared_50/causal_num_{num_causal}/"
            data_dir = os.path.join(wrk_dir, "summary_data/")
            
            
            # Reading the data
            zfile_path = f"{data_dir}CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{int(h2_num)}"
            try:
                zfile = np.genfromtxt(zfile_path, delimiter=' ', names=True, dtype=None, encoding='utf-8')
            except Exception as e:
                print(f"Error reading zfile for CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{h2_num}: {e}")
                continue  # Skip to next iteration if file not found or other error occurs
            
            zscore_1 = zfile['zscore_1']
            zscore_2 = zfile['zscore_2']
            N_list = [300000, 300000]
            z_list = [zscore_1, zscore_2]
            
            # Reading covariance matrices
            try:
                EU_cov = np.genfromtxt(f"{data_dir}CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{int(h2_num)}.LD1", delimiter=' ')
                BB_cov = np.genfromtxt(f"{data_dir}CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{int(h2_num)}.LD2", delimiter=' ')
            except Exception as e:
                print(f"Error reading covariance matrices for CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{h2_num}: {e}")
                continue
            
            R_list = [EU_cov, BB_cov]
            
            # Reading MAF data
            try:
                maf_EU = np.loadtxt(f'/scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_1mb/risk_loci_ld_eur/maf_loci_{LD_BLOCK}_maf.txt')
                maf_BB = np.loadtxt(f'/scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_1mb/risk_loci_ld_afr/maf_loci_{LD_BLOCK}_maf.txt')
            except Exception as e:
                print(f"Error reading MAF data for LD_BLOCK_{LD_BLOCK}: {e}")
                continue
            
            maf_list = [maf_EU, maf_BB]
            
            # Record start time
            start_time = time.time()
            
            # Run MultiSuSiE
            try:
                ss_fit = MultiSuSiE.multisusie_rss(
                    z_list=z_list,
                    R_list=R_list,
                    rho=np.array([[1, 0.8], [0.8, 1]]),
                    population_sizes=N_list,
                    L=10,
                    scaled_prior_variance=0.2,
                    low_memory_mode=False,
                    min_abs_corr=0.5,
                    single_population_mac_thresh=20,
                    maf_list=maf_list,
                    coverage=0.95
                )
            except Exception as e:
                print(f"Error running MultiSuSiE for CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{h2_num}: {e}")
                continue
            
            # Record end time and calculate duration
            end_time = time.time()
            time_taken = (end_time - start_time) / 60
            
            # Define file paths for saving results
            file_path = f'/scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_1mb/shared_50/multi_susie_result/MultiSuSiE_CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{h2_num}_output'
            
            # Save the run time to a text file
            runtime_file = f'{file_path}_runtime.txt'
            with open(runtime_file, 'w') as f:
                f.write(f"Time taken: {time_taken:.8f} minutes\n")
            
            # Convert structured array to DataFrame
            df_zfile = pd.DataFrame(zfile)
            
            # Add the ss_fit.pip values as a new column in the DataFrame
            df_zfile['pip'] = ss_fit.pip
            
            # Filter significant credible sets
            filtered_sets = [ss_fit.sets[0][i] for i in range(len(ss_fit.sets[0])) if ss_fit.sets[3][i] == True]
            
            # Create a new empty DataFrame to store the results
            filtered_df = pd.DataFrame()
            
            # For each filtered set, find corresponding SNPs by their index in zfile and label with CS index
            for idx, cs_set in enumerate(filtered_sets):
                for snp_index in cs_set:
                    # Select the row in df_zfile based on the SNP index
                    if snp_index < len(df_zfile):
                        matching_snp_df = df_zfile.iloc[[snp_index]].copy()
                        # Add the CS index to the matched SNP
                        matching_snp_df['CS'] = idx + 1
                        # Append the matching SNP data to the new filtered DataFrame
                        filtered_df = pd.concat([filtered_df, matching_snp_df])
            
            # Define file names for txt files
            zfile_output = f'{file_path}_snp.txt'
            filtered_output = f'{file_path}_cs.txt'
            
            # Save the df_zfile DataFrame to a .txt file with tab separation
            df_zfile.to_csv(zfile_output, sep='\t', index=False)
            
            # Save the filtered_df DataFrame to a .txt file with tab separation
            filtered_df.to_csv(filtered_output, sep='\t', index=False)


Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censor

In [2]:
import os
import numpy as np
import pandas as pd
import time
import MultiSuSiE
import resource
from multiprocessing import Process # Import Process

# --- 1. Define the worker function ---
# Move the simulation logic inside this function
def run_simulation(num_causal, LD_BLOCK, h2_num):
    # Set directories and file paths
    wrk_dir = f"/scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_1mb/shared_50/causal_num_{num_causal}/"
    data_dir = os.path.join(wrk_dir, "summary_data/")
    
    # Data-reading section
    
    # (Abbreviated data-reading section)
    zfile_path = f"{data_dir}CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{int(h2_num)}"
    try:
        zfile = np.genfromtxt(zfile_path, delimiter=' ', names=True, dtype=None, encoding='utf-8')
        zscore_1 = zfile['zscore_1']
        zscore_2 = zfile['zscore_2']
        N_list = [300000, 300000]
        z_list = [zscore_1, zscore_2]
        
        EU_cov = np.genfromtxt(f"{data_dir}CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{int(h2_num)}.LD1", delimiter=' ')
        BB_cov = np.genfromtxt(f"{data_dir}CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{int(h2_num)}.LD2", delimiter=' ')
        R_list = [EU_cov, BB_cov]
        
        maf_EU = np.loadtxt(f'/scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_1mb/risk_loci_ld_eur/maf_loci_{LD_BLOCK}_maf.txt')
        maf_BB = np.loadtxt(f'/scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_1mb/risk_loci_ld_afr/maf_loci_{LD_BLOCK}_maf.txt')
        maf_list = [maf_EU, maf_BB]

        # Record start time
        start_time = time.time()

        # Run MultiSuSiE
        ss_fit = MultiSuSiE.multisusie_rss(
            z_list=z_list,
            R_list=R_list,
            rho=np.array([[1, 0.8], [0.8, 1]]),
            population_sizes=N_list,
            L=10,
            scaled_prior_variance=0.2,
            low_memory_mode=False,
            min_abs_corr=0.5,
            single_population_mac_thresh=20,
            maf_list=maf_list,
            coverage=0.95
        )
        
        # 1. Calculate Time
        end_time = time.time()
        elapsed_sec = end_time - start_time
        time_taken_min = elapsed_sec / 60
        
        # 2. Calculate Peak Memory (Now valid because process is fresh)
        usage_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        peak_mem_mib = usage_kb / 1024

        # 3. Create DataFrame and Save
        perf_data = {
            'Method': ['MultiSuSiE'],
            'Time_Min': [time_taken_min],
            'Elapsed_Time_sec': [elapsed_sec],
            'Peak_RAM_Used_MiB': [peak_mem_mib]
        }
        
        df_perf = pd.DataFrame(perf_data)
        file_path = f'/scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_1mb/shared_50/multi_susie_result/runtime/MultiSuSiE_CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{h2_num}_output'
        runtime_file = f'{file_path}_runtime_memory.txt'
        df_perf.to_csv(runtime_file, sep='\t', index=False)
        
    except Exception as e:
        print(f"Error in CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{h2_num}: {e}")

# --- 2. Main Loop ---
if __name__ == "__main__":
    num_causal_range = range(3, 4)
    LD_BLOCK_range = range(89, 101)
    h2_num_range = range(1, 3)

    for num_causal in num_causal_range:
        for LD_BLOCK in LD_BLOCK_range:
            for h2_num in h2_num_range:
                
                # Create a separate process for this iteration
                p = Process(target=run_simulation, args=(num_causal, LD_BLOCK, h2_num))
                p.start()
                p.join() # Wait for it to finish before starting the next (Serial execution)

Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censor

In [2]:
import os
import numpy as np
import pandas as pd
import time
import MultiSuSiE

# Define ranges for num_causal, LD_BLOCK, and h2_num
num_causal_range = range(1, 4)  # 1 to 3 inclusive
LD_BLOCK_range = range(1, 101)  # 1 to 100 inclusive
h2_num_range = range(1, 3)      # 1 to 2 inclusive

# Loop through num_causal, LD_BLOCK, and h2_num
for num_causal in num_causal_range:
    for LD_BLOCK in LD_BLOCK_range:
        for h2_num in h2_num_range:
            
            # Set directories and file paths
            wrk_dir = f"/scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_1mb/shared_all/causal_num_{num_causal}/"
            data_dir = os.path.join(wrk_dir, "summary_data/")
            
            
            # Reading the data
            zfile_path = f"{data_dir}CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{int(h2_num)}"
            try:
                zfile = np.genfromtxt(zfile_path, delimiter=' ', names=True, dtype=None, encoding='utf-8')
            except Exception as e:
                print(f"Error reading zfile for CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{h2_num}: {e}")
                continue  # Skip to next iteration if file not found or other error occurs
            
            zscore_1 = zfile['zscore_1']
            zscore_2 = zfile['zscore_2']
            N_list = [300000, 300000]
            z_list = [zscore_1, zscore_2]
            
            # Reading covariance matrices
            try:
                EU_cov = np.genfromtxt(f"{data_dir}CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{int(h2_num)}.LD1", delimiter=' ')
                BB_cov = np.genfromtxt(f"{data_dir}CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{int(h2_num)}.LD2", delimiter=' ')
            except Exception as e:
                print(f"Error reading covariance matrices for CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{h2_num}: {e}")
                continue
            
            R_list = [EU_cov, BB_cov]
            
            # Reading MAF data
            try:
                maf_EU = np.loadtxt(f'/scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_1mb/risk_loci_ld_eur/maf_loci_{LD_BLOCK}_maf.txt')
                maf_BB = np.loadtxt(f'/scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_1mb/risk_loci_ld_afr/maf_loci_{LD_BLOCK}_maf.txt')
            except Exception as e:
                print(f"Error reading MAF data for LD_BLOCK_{LD_BLOCK}: {e}")
                continue
            
            maf_list = [maf_EU, maf_BB]
            
            # Record start time
            start_time = time.time()
            
            # Run MultiSuSiE
            try:
                ss_fit = MultiSuSiE.multisusie_rss(
                    z_list=z_list,
                    R_list=R_list,
                    rho=np.array([[1, 0.8], [0.8, 1]]),
                    population_sizes=N_list,
                    L=10,
                    scaled_prior_variance=0.2,
                    low_memory_mode=False,
                    min_abs_corr=0.5,
                    single_population_mac_thresh=20,
                    maf_list=maf_list,
                    coverage=0.95
                )
            except Exception as e:
                print(f"Error running MultiSuSiE for CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{h2_num}: {e}")
                continue
            
            # Record end time and calculate duration
            end_time = time.time()
            time_taken = (end_time - start_time) / 60
            
            # Define file paths for saving results
            file_path = f'/scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_1mb/shared_all/multi_susie_result/MultiSuSiE_CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{h2_num}_output'
            
            # Save the run time to a text file
            runtime_file = f'{file_path}_runtime.txt'
            with open(runtime_file, 'w') as f:
                f.write(f"Time taken: {time_taken:.8f} minutes\n")
            
            # Convert structured array to DataFrame
            df_zfile = pd.DataFrame(zfile)
            
            # Add the ss_fit.pip values as a new column in the DataFrame
            df_zfile['pip'] = ss_fit.pip
            
            # Filter significant credible sets
            filtered_sets = [ss_fit.sets[0][i] for i in range(len(ss_fit.sets[0])) if ss_fit.sets[3][i] == True]
            
            # Create a new empty DataFrame to store the results
            filtered_df = pd.DataFrame()
            
            # For each filtered set, find corresponding SNPs by their index in zfile and label with CS index
            for idx, cs_set in enumerate(filtered_sets):
                for snp_index in cs_set:
                    # Select the row in df_zfile based on the SNP index
                    if snp_index < len(df_zfile):
                        matching_snp_df = df_zfile.iloc[[snp_index]].copy()
                        # Add the CS index to the matched SNP
                        matching_snp_df['CS'] = idx + 1
                        # Append the matching SNP data to the new filtered DataFrame
                        filtered_df = pd.concat([filtered_df, matching_snp_df])
            
            # Define file names for txt files
            zfile_output = f'{file_path}_snp.txt'
            filtered_output = f'{file_path}_cs.txt'
            
            # Save the df_zfile DataFrame to a .txt file with tab separation
            df_zfile.to_csv(zfile_output, sep='\t', index=False)
            
            # Save the filtered_df DataFrame to a .txt file with tab separation
            filtered_df.to_csv(filtered_output, sep='\t', index=False)


Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC
Censor